# Random Forest Task

In [66]:
# Import Libraries
import pandas as pd
from scipy import stats
import seaborn as sns 
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier

In [100]:
# Load the Titanic dataset from a CSV file into a pandas DataFrame
titanic_df = pd.read_csv("Titanic.csv")

# Display the first few rows of the DataFrame to get a preliminary look at the data
titanic_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [104]:
# get the number of missing data points per column
missing_values_count = titanic_df.isnull().sum()

# Look at the number of missing points per column
missing_values_count

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [106]:
# Print a summary of the Titanic DataFrame, including data types and missing values.
print(titanic_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [108]:
# Checks for duplicate rows within the titanic_df DataFrame.
duplicate_data = titanic_df[titanic_df.duplicated()] 

duplicate_data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked


In [110]:
# Remove Cabin and Age variables as there are a massive missing values
titanic_df.drop(["Cabin", "Age"], axis=1, inplace=True)

In [112]:
# Drop the columns, as it's not needed for analysis. 
titanic_df.drop(["PassengerId", "Ticket", "Embarked"], axis=1, inplace=True)

titanic_df.head()

,Survived,Pclass,Name,Sex,SibSp,Parch,Fare
0,0,3,"Braund, Mr. Owen Harris",male,1,0,7.2500
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,1,0,71.2833
2,1,3,"Heikkinen, Miss. Laina",female,0,0,7.9250
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,1,0,53.1000
4,0,3,"Allen, Mr. William Henry",male,0,0,8.0500


In [114]:
# Apply one-hot encoding to the 'Sex' column
titanic_df = pd.get_dummies(titanic_df, prefix="Sex", columns=["Sex"])
titanic_df.head()

,Survived,Pclass,Name,SibSp,Parch,Fare,Sex_female,Sex_male
0,0,3,"Braund, Mr. Owen Harris",1,0,7.2500,False,True
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,71.2833,True,False
2,1,3,"Heikkinen, Miss. Laina",0,0,7.9250,True,False
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,53.1000,True,False
4,0,3,"Allen, Mr. William Henry",0,0,8.0500,False,True


In [116]:
# Select feature columns (all except 'Survived')
X = titanic_df.iloc[:,[1,3,4,5,6,7]].values

# Select the target columns ('Survived')
y = titanic_df.iloc[:,[0]].values

# Split data into training and testing sets (75% train, 25% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [118]:
forest = RandomForestClassifier(n_estimators=100, random_state=7)
forest.fit(X_train, y_train.ravel())

feature_imp = pd.Series(forest.feature_importances_).sort_values(ascending=False)
feature_imp

3    0.453696
5    0.173958
4    0.136133
0    0.097752
1    0.071289
2    0.067172
dtype: float64

Based on the provided feature importance, the first selected column, PassengerId, in the original Titanic DataFrame contributes the most to predicting survival.

In [120]:
# Base Decision Tree (for bagging and boosting)
base = DecisionTreeClassifier(max_depth=5, random_state=42)
base.fit(X_train, y_train.ravel())
base_accuracy = base.score(X_test, y_test)
print(f"Base Decision Tree Accuracy: {base_accuracy}")

# Bagging Classifier
ensemble = BaggingClassifier(estimator=base, n_estimators=50, random_state=5)
ensemble.fit(X_train, y_train.ravel())
ensemble_accuracy = ensemble.score(X_test, y_test)
print(f"Bagging Classifier Accuracy: {ensemble_accuracy}")

# AdaBoost Classifier
adaboost_clf = AdaBoostClassifier(estimator=base, n_estimators=50, random_state=7)
adaboost_clf.fit(X_train, y_train.ravel())
adaboost_accuracy = adaboost_clf.score(X_test, y_test)
print(f"AdaBoost Classifier Accuracy: {adaboost_accuracy}")

Base Decision Tree Accuracy: 0.8116591928251121
Bagging Classifier Accuracy: 0.820627802690583
AdaBoost Classifier Accuracy: 0.8026905829596412


In [ ]:
# Tuned Random Forest Model

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 5, 7, 10, None]
}
# Initialize a RandomForestClassifier
forest = RandomForestClassifier(random_state=7)

# Initialize GridSearchCV to perform hyperparameter tuning.
grid_search = GridSearchCV(estimator=forest, param_grid=param_grid, cv=5, scoring='accuracy') 
grid_search.fit(X_train, y_train.ravel())

# Get the best trained RandomForestClassifier from GridSearchCV.
best_forest = grid_search.best_estimator_

# Evaluate the best model on the test set and get the accuracy.
tuned_forest_accuracy = best_forest.score(X_test, y_test) 

# Get the best hyperparameter values found by GridSearchCV.
best_params = grid_search.best_params_

# Print the accuracy of the tuned Random Forest model.
print(f"Tuned Random Forest Accuracy: {tuned_forest_accuracy}") 
# Print the best hyperparameter values.
print(f"Tuned Random Forest Best Parameters: {best_params}") 

# Determine the best model
models = {
    "Base Decision Tree": base_accuracy,  # Store the accuracy of the base Decision Tree model.
    "Bagging Classifier": ensemble_accuracy,  # Store the accuracy of the Bagging Classifier model.
    "Tuned Random Forest": tuned_forest_accuracy  # Store the accuracy of the tuned Random Forest model.
}
# Find the model with the highest accuracy.
best_model_name = max(models, key=models.get)
# Get the accuracy of the best model.
best_model_accuracy = models[best_model_name]

# Print the name of the best performing model.
print(f"\nThe best performing model is: {best_model_name}") 
# Print the accuracy of the best performing model.
print(f"Best Model Accuracy: {best_model_accuracy}")  

if best_model_name == "Tuned Random Forest":  # Check if the best model is the tuned Random Forest.
    print(f"Best Model n_estimators: {best_params['n_estimators']}")  # Print the best n_estimators value.
    print(f"Best Model max_depth: {best_params['max_depth']}")  # Print the best max_depth value.